# STALGCM Deliverable

## Problem Set 2: Text Cleaning

### Members:
- Jeff Uriel V. Fonte
- Ulrich Maeko L. Gonzales

Import libraries

In [1]:
import re
import requests
from pathlib import Path
from bs4 import BeautifulSoup

URLs

In [2]:
urls = ['https://en.wikipedia.org/wiki/2016_Philippine_presidential_election',
        'https://en.wikipedia.org/wiki/2022_Philippine_presidential_election']

In [3]:
headers = {
    'User-Agent': 'Mozilla/5.0'
    }

Web Parser with BeautifulSoup

In [ ]:
def extract_text(url, headers):
    page = requests.get(url, headers=headers)
    if page.status_code != 200:
        return None

    soup = BeautifulSoup(page.content, 'html.parser')
    body = soup.find('div', class_='mw-parser-output')
    if not body:
        return None

    ui_selectors = [
        'table', '.infobox', '.navbox', '.vertical-navbox',
        '.sidebar', '.hatnote', '.mw-editsection', '.metadata',
        '.toc', '.catlinks'
    ]
    for element in body.select(','.join(ui_selectors)):
        element.decompose()

    content = []
    targets = body.find_all(['p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'ul', 'ol'])

    for element in targets:
        if element.name in ['ul', 'ol']:
            for li in element.find_all('li', recursive=False):
                raw_str = li.get_text()
                text = ' '.join(raw_str.split())
                if text:
                    content.append(text)
        else:
            raw_str = element.get_text()
            text = ' '.join(raw_str.split())
            if text:
                content.append(text)

    return '\n\n'.join(content)

In [5]:
def save_text(file_path, text):
    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_text(text, encoding='utf-8')

RegEx

In [6]:
def clean_text(text):
    text = re.sub(r'\s+([.,;:!?])', r'\1', text)
    text = re.sub(r'\(\s+', '(', text)
    text = re.sub(r'\s+\)', ')', text)
    return text

In [ ]:
for url in urls:
    year = "2016" if "2016" in url else "2022"
    raw = Path(f"data/raw/{year}")
    raw.mkdir(parents=True, exist_ok=True)

    raw_text = extract_text(url, headers)

    file_path = raw / f"raw_{year}_election.txt"
    save_text(file_path, raw_text)

    # RegEx cleaning
    processed_text = clean_text(raw_text)
    output_dir = Path('output')
    file_path = output_dir / f"processed_{year}_election.txt"
    save_text(file_path, processed_text)